In [ ]:
# This notebook explores the NRN GPKG files to understand schema differences across provinces and territories
# With the goal of generalizing data access for subsequent analysis. Key findings include:
# 1. All provinces and territories contain a ROADSEG table.
# 2. Table names differ due to version numbers and should not be hardcoded.
# 3. A common set of helper functions can abstract away province-specific naming.
# 4. ROADCLASS provides a natural mechanism for reducing network size.
# 5. Freeway, Expressway / Highway, Ramp, and Arterial classes constitute a 
#    plausible minimum viable freight road network representation.
# 6. These abstractions enable construction of a unified Canadian road network in subsequent notebooks.

from pathlib import Path
import sqlite3

import pandas as pd

# Define the path to the raw NRN data directory

raw_nrn = Path(
    r"C:\Users\aviga\Research\repos\temoa_geospace\data_files\raw\nrn"
)

In [ ]:
# Define the province and territory abbreviations

provinces = [
    "AB", "BC", "MB", "NB", "NL", "NS",
    "NT", "NU", "ON", "PE", "QC", "SK", "YT"
]

# Function to get the path to the English GPKG file for a given province

def get_nrn_gpkg_path(province: str) -> Path:

    province = province.upper()
    province_dir = raw_nrn / province

    matches = list(province_dir.glob("*_en.gpkg"))

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one English GPKG file for {province}, "
            f"found {len(matches)}."
        )

    return matches[0]

# Verify that each province and territory has a single English GPKG file

for province in provinces:
    gpkg_path = get_nrn_gpkg_path(province)

    print(
        f"{province}: {gpkg_path.name}"
    )

# Note:
# NRN GeoPackage version numbers differ across provinces and territories.
# Filenames should never be hardcoded. Instead, files are located dynamically using wildcard matching on '*_en.gpkg'.

AB: NRN_AB_17_0_GPKG_en.gpkg
BC: NRN_BC_14_0_GPKG_en.gpkg
MB: NRN_MB_6_0_GPKG_en.gpkg
NB: NRN_NB_15_0_GPKG_en.gpkg
NL: NRN_NL_7_0_GPKG_en.gpkg
NS: NRN_NS_18_0_GPKG_en.gpkg
NT: NRN_NT_16_0_GPKG_en.gpkg
NU: NRN_NU_13_0_GPKG_en.gpkg
ON: NRN_ON_18_0_GPKG_en.gpkg
PE: NRN_PE_22_1_GPKG_en.gpkg
QC: NRN_QC_9_0_GPKG_en.gpkg
SK: NRN_SK_15_0_GPKG_en.gpkg
YT: NRN_YT_20_0_GPKG_en.gpkg


In [5]:
# Function to list all layers in a GeoPackage

def get_gpkg_contents(gpkg_path: Path) -> pd.DataFrame:

    with sqlite3.connect(gpkg_path) as conn:

        contents = pd.read_sql_query(
            """
            SELECT *
            FROM gpkg_contents
            """,
            conn
        )

    return contents

# Examine Ontario contents

contents_on = get_gpkg_contents(
    get_nrn_gpkg_path("ON")
)

contents_on

,table_name,data_type,identifier,description,last_change,min_x,min_y,max_x,max_y,srs_id
0,NRN_ON_18_0_STRPLANAME,attributes,NRN_ON_18_0_STRPLANAME,,2025-02-03T14:53:41.198Z,NaN,NaN,NaN,NaN,0
1,NRN_ON_18_0_TOLLPOINT,features,NRN_ON_18_0_TOLLPOINT,,2025-02-03T14:53:41.211Z,-83.036431,41.763366,-75.980277,44.368237,4617
2,NRN_ON_18_0_FERRYSEG,features,NRN_ON_18_0_FERRYSEG,,2025-02-03T14:53:41.221Z,-93.827784,41.676546,-74.882287,51.279179,4617
3,NRN_ON_18_0_JUNCTION,features,NRN_ON_18_0_JUNCTION,,2025-02-03T14:53:43.762Z,-95.153425,41.676546,-74.343857,56.082803,4617
4,NRN_ON_18_0_ADDRANGE,attributes,NRN_ON_18_0_ADDRANGE,,2025-02-03T14:53:43.766Z,NaN,NaN,NaN,NaN,0
5,NRN_ON_18_0_ROADSEG,features,NRN_ON_18_0_ROADSEG,,2025-02-03T14:53:47.051Z,-95.153425,41.734662,-74.343857,56.082803,4617
6,NRN_ON_18_0_BLKPASSAGE,features,NRN_ON_18_0_BLKPASSAGE,,2025-02-03T14:53:47.076Z,-95.091639,41.931241,-74.363732,52.219129,4617


In [6]:
for province in provinces:

    gpkg_path = get_nrn_gpkg_path(province)

    contents = get_gpkg_contents(gpkg_path)

    print(f"\n{province}")
    print(contents["table_name"].tolist())


AB
['NRN_AB_17_0_BLKPASSAGE', 'NRN_AB_17_0_FERRYSEG', 'NRN_AB_17_0_STRPLANAME', 'NRN_AB_17_0_TOLLPOINT', 'NRN_AB_17_0_ROADSEG', 'NRN_AB_17_0_JUNCTION', 'NRN_AB_17_0_ADDRANGE']

BC
['NRN_BC_14_0_BLKPASSAGE', 'NRN_BC_14_0_FERRYSEG', 'NRN_BC_14_0_JUNCTION', 'NRN_BC_14_0_ROADSEG', 'NRN_BC_14_0_TOLLPOINT']

MB
['NRN_MB_6_0_BLKPASSAGE', 'NRN_MB_6_0_FERRYSEG', 'NRN_MB_6_0_JUNCTION', 'NRN_MB_6_0_ROADSEG', 'NRN_MB_6_0_TOLLPOINT']

NB
['NRN_NB_15_0_STRPLANAME', 'NRN_NB_15_0_ADDRANGE', 'NRN_NB_15_0_JUNCTION', 'NRN_NB_15_0_ROADSEG', 'NRN_NB_15_0_FERRYSEG']

NL
['NRN_NL_7_0_BLKPASSAGE', 'NRN_NL_7_0_FERRYSEG', 'NRN_NL_7_0_JUNCTION', 'NRN_NL_7_0_ROADSEG']

NS
['NRN_NS_18_0_BLKPASSAGE', 'NRN_NS_18_0_ADDRANGE', 'NRN_NS_18_0_JUNCTION', 'NRN_NS_18_0_ROADSEG', 'NRN_NS_18_0_STRPLANAME', 'NRN_NS_18_0_FERRYSEG', 'NRN_NS_18_0_TOLLPOINT']

NT
['NRN_NT_16_0_STRPLANAME', 'NRN_NT_16_0_BLKPASSAGE', 'NRN_NT_16_0_JUNCTION', 'NRN_NT_16_0_FERRYSEG', 'NRN_NT_16_0_ROADSEG', 'NRN_NT_16_0_ADDRANGE']

NU
['NRN_NU_13_0_STR

In [7]:
# Function to get the road segment table name from a GeoPackage

def get_roadseg_table(gpkg_path: Path) -> str:

    contents = get_gpkg_contents(gpkg_path)

    matches = contents[
        contents["table_name"].str.endswith("_ROADSEG")
    ]["table_name"].tolist()

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one ROADSEG table in {gpkg_path.name}, "
            f"found {len(matches)}: {matches}"
        )

    return matches[0]

# Verify road segment table names for all provinces and territories

for province in provinces:
    gpkg_path = get_nrn_gpkg_path(province)
    roadseg_table = get_roadseg_table(gpkg_path)

    print(f"{province}: {roadseg_table}")

AB: NRN_AB_17_0_ROADSEG
BC: NRN_BC_14_0_ROADSEG
MB: NRN_MB_6_0_ROADSEG
NB: NRN_NB_15_0_ROADSEG
NL: NRN_NL_7_0_ROADSEG
NS: NRN_NS_18_0_ROADSEG
NT: NRN_NT_16_0_ROADSEG
NU: NRN_NU_13_0_ROADSEG
ON: NRN_ON_18_0_ROADSEG
PE: NRN_PE_22_1_ROADSEG
QC: NRN_QC_9_0_ROADSEG
SK: NRN_SK_15_0_ROADSEG
YT: NRN_YT_20_0_ROADSEG


In [8]:
# Function to summarize ROADCLASS counts for one province or territory

def get_roadclass_counts(province: str) -> pd.DataFrame:

    gpkg_path = get_nrn_gpkg_path(province)
    roadseg_table = get_roadseg_table(gpkg_path)

    with sqlite3.connect(gpkg_path) as conn:
        counts = pd.read_sql_query(
            f"""
            SELECT ROADCLASS,
                   COUNT(*) AS segments
            FROM {roadseg_table}
            GROUP BY ROADCLASS
            ORDER BY segments DESC;
            """,
            conn
        )

    counts.insert(0, "province", province)

    return counts

# Build ROADCLASS summary for all provinces and territories

roadclass_counts_all = pd.concat(
    [get_roadclass_counts(province) for province in provinces],
    ignore_index=True
)

roadclass_counts_all

,province,ROADCLASS,segments
0,AB,Collector,170670
1,AB,Local / Street,162403
2,AB,Resource / Recreation,46188
3,AB,Arterial,28920
4,AB,Ramp,13129
...,...,...,...
134,YT,Resource / Recreation,111
135,YT,Service Lane,96
136,YT,Ramp,29
137,YT,Winter,2


In [9]:
# Pivot the ROADCLASS summary to have provinces as rows and ROADCLASS categories as columns

roadclass_pivot = roadclass_counts_all.pivot_table(
    index="province",
    columns="ROADCLASS",
    values="segments",
    fill_value=0,
    aggfunc="sum"
)

roadclass_pivot

ROADCLASS,Alleyway / Lane,Arterial,Collector,Expressway / Highway,Freeway,Local / Strata,Local / Street,Local / Unknown,Ramp,Rapid Transit,Resource / Recreation,Service Lane,Unknown,Winter
province,,,,,,,,,,,,,,
AB,201,28920,170670,7344,170,12074,162403,1208,13129,110,46188,1136,0,40
BC,12084,18923,30309,9741,1843,15179,157655,3701,3686,0,6873,3590,0,0
MB,0,2,44939,20857,59,244,42902,0,1449,7,12,57,0,76
NB,7,441,6626,2031,1168,10,44939,8700,1930,0,3638,205,19,0
NL,4,19,12984,7247,465,108,22774,87,724,0,63,9,0,0
NS,0,7557,6910,1845,1038,0,68736,0,1479,0,31879,324,78,0
NT,223,0,0,981,0,0,0,5377,3,0,1219,55,344,338
NU,0,0,135,0,0,0,252,30,0,0,0,67,4653,0
ON,6904,96835,53480,12510,6903,61875,357511,850,7551,558,45681,767,3,234


In [11]:
# Define ROADCLASS categories for freight access and backbone

backbone_classes = [
    "Freeway",
    "Expressway / Highway",
    "Ramp",
]

freight_access_classes = backbone_classes + [
    "Arterial",
]